In [ ]:
from pathlib import Path
import os
import zipfile

archive_path = Path("/content/lab04.zip")
if not archive_path.exists():
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        pass
if archive_path.exists():
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall("/content")
project_path = Path("/content/lab04")
if project_path.exists() and (project_path / "preprocessing.py").exists():
    os.chdir(project_path)
%pip install -q -r requirements.txt

In [ ]:
import json
import time
import html
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from preprocessing import normalize_text, inspect_split, stratified_sample, remove_test_overlap, average_word2vec, binary_metrics

SEED = 42
TRAIN_SAMPLE_SIZE = 10000
TEST_SAMPLE_SIZE = 2500
WORD2VEC_SIZE = 100
OUTPUT_DIR = Path('.')
np.random.seed(SEED)

In [ ]:
dataset = load_dataset("stanfordnlp/imdb", "plain_text")
train_full = dataset["train"].to_pandas()[["text", "label"]]
test_full = dataset["test"].to_pandas()[["text", "label"]]

label_names = {0: "negative", 1: "positive"}
display(pd.DataFrame([
    {"Split": "Train", "Samples": len(train_full), "Classes": train_full["label"].nunique()},
    {"Split": "Test", "Samples": len(test_full), "Classes": test_full["label"].nunique()},
]))

In [ ]:
inspection = pd.DataFrame([
    inspect_split(train_full, "train"),
    inspect_split(test_full, "test"),
])
display(inspection)

class_rows = []
for split_name, frame in [("train", train_full), ("test", test_full)]:
    counts = frame["label"].value_counts().sort_index()
    for label, count in counts.items():
        class_rows.append({
            "Split": split_name,
            "Class": label_names[int(label)],
            "Count": int(count),
            "Percentage": round(100 * count / len(frame), 2),
        })
class_distribution = pd.DataFrame(class_rows)
display(class_distribution)

fig, ax = plt.subplots(figsize=(7, 4))
plot_data = class_distribution.pivot(index="Class", columns="Split", values="Count")
plot_data.plot(kind="bar", ax=ax)
ax.set_title("IMDb class distribution")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Samples")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig("dataset_class_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

length_rows = []
for split_name, frame in [("train", train_full), ("test", test_full)]:
    word_lengths = frame["text"].fillna("").map(lambda value: len(simple_preprocess(str(value), deacc=True)))
    length_rows.append({
        "Split": split_name,
        "Mean words": round(word_lengths.mean(), 2),
        "Median words": round(word_lengths.median(), 2),
        "95th percentile": round(word_lengths.quantile(0.95), 2),
        "Maximum words": int(word_lengths.max()),
    })
display(pd.DataFrame(length_rows))

train_normalized = set(train_full["text"].map(normalize_text))
test_normalized = set(test_full["text"].map(normalize_text))
cross_split_duplicates = len(train_normalized.intersection(test_normalized))
display(pd.DataFrame([{"Exact normalized documents shared by official train and test": cross_split_duplicates}]))

In [ ]:
train_sample = stratified_sample(train_full, TRAIN_SAMPLE_SIZE, random_state=SEED)
test_sample = stratified_sample(test_full, TEST_SAMPLE_SIZE, random_state=SEED + 1)
test_sample, removed_test_duplicates = remove_test_overlap(train_full, test_sample)

X_train, X_val, y_train, y_val = train_test_split(
    train_sample["text"].astype(str),
    train_sample["label"].astype(int),
    test_size=0.2,
    random_state=SEED,
    stratify=train_sample["label"],
)
X_test = test_sample["text"].astype(str)
y_test = test_sample["label"].astype(int)

split_summary = pd.DataFrame([
    {"Split": "train", "Samples": len(X_train), "Positive percentage": round(100 * y_train.mean(), 2)},
    {"Split": "validation", "Samples": len(X_val), "Positive percentage": round(100 * y_val.mean(), 2)},
    {"Split": "test", "Samples": len(X_test), "Positive percentage": round(100 * y_test.mean(), 2)},
])
display(split_summary)
display(pd.DataFrame([{"Duplicate test samples removed": removed_test_duplicates}]))

## 5. Experiment 1 — TF-IDF + Logistic Regression


In [ ]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 1),
        min_df=2,
        max_features=50000,
        sublinear_tf=True,
    )),
    ("classifier", LogisticRegression(max_iter=1500, random_state=SEED)),
])

start_time = time.perf_counter()
baseline.fit(X_train, y_train)
baseline_training_time = time.perf_counter() - start_time
baseline_test_predictions = baseline.predict(X_test)
baseline_validation_probabilities = baseline.predict_proba(X_val)[:, 1]
baseline_vocabulary_size = len(baseline.named_steps["tfidf"].vocabulary_)
baseline_metrics = binary_metrics(y_test, baseline_test_predictions)

display(pd.DataFrame([{
    "Model": "TF-IDF + Logistic Regression",
    "Training time seconds": round(baseline_training_time, 3),
    "Vocabulary size": baseline_vocabulary_size,
    **{key: round(value, 4) if isinstance(value, float) else value for key, value in baseline_metrics.items()},
}]))

## 6. Prediction threshold


In [ ]:
threshold_rows = []
for threshold in [0.5, 0.8]:
    validation_predictions = (baseline_validation_probabilities >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val, validation_predictions),
        "Precision": precision_score(y_val, validation_predictions, zero_division=0),
        "Recall": recall_score(y_val, validation_predictions, zero_division=0),
        "F1": f1_score(y_val, validation_predictions, zero_division=0),
        "Predicted positive": int(validation_predictions.sum()),
    })
threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results.round(4))

## 7. Experiment 2 — Average Word2Vec + Logistic Regression



In [ ]:
training_tokens = [simple_preprocess(normalize_text(text), deacc=True) for text in X_train]

start_time = time.perf_counter()
word2vec_model = Word2Vec(
    sentences=training_tokens,
    vector_size=WORD2VEC_SIZE,
    window=5,
    min_count=2,
    workers=1,
    sg=1,
    epochs=10,
    seed=SEED,
)
X_train_w2v = average_word2vec(X_train.tolist(), word2vec_model, WORD2VEC_SIZE)
X_val_w2v = average_word2vec(X_val.tolist(), word2vec_model, WORD2VEC_SIZE)
X_test_w2v = average_word2vec(X_test.tolist(), word2vec_model, WORD2VEC_SIZE)
w2v_classifier = LogisticRegression(max_iter=1500, random_state=SEED)
w2v_classifier.fit(X_train_w2v, y_train)
w2v_training_time = time.perf_counter() - start_time
w2v_test_predictions = w2v_classifier.predict(X_test_w2v)
w2v_metrics = binary_metrics(y_test, w2v_test_predictions)
w2v_vocabulary_size = len(word2vec_model.wv)

display(pd.DataFrame([{
    "Model": "Average Word2Vec + Logistic Regression",
    "Training time seconds": round(w2v_training_time, 3),
    "Word2Vec vocabulary size": w2v_vocabulary_size,
    "Feature dimension": WORD2VEC_SIZE,
    **{key: round(value, 4) if isinstance(value, float) else value for key, value in w2v_metrics.items()},
}]))

## 8. Experiment 3 — TF-IDF + Linear SVM



In [ ]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 1),
        min_df=2,
        max_features=50000,
        sublinear_tf=True,
    )),
    ("classifier", LinearSVC(C=1.0, random_state=SEED)),
])

start_time = time.perf_counter()
svm_pipeline.fit(X_train, y_train)
svm_training_time = time.perf_counter() - start_time
svm_test_predictions = svm_pipeline.predict(X_test)
svm_vocabulary_size = len(svm_pipeline.named_steps["tfidf"].vocabulary_)
svm_metrics = binary_metrics(y_test, svm_test_predictions)

display(pd.DataFrame([{
    "Model": "TF-IDF + Linear SVM",
    "Training time seconds": round(svm_training_time, 3),
    "Vocabulary size": svm_vocabulary_size,
    **{key: round(value, 4) if isinstance(value, float) else value for key, value in svm_metrics.items()},
}]))

## 9. Experiment 4 — Unigram vs unigram + bigram



In [ ]:
bigram_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_features=80000,
        sublinear_tf=True,
    )),
    ("classifier", LogisticRegression(max_iter=1500, random_state=SEED)),
])

start_time = time.perf_counter()
bigram_pipeline.fit(X_train, y_train)
bigram_training_time = time.perf_counter() - start_time
bigram_test_predictions = bigram_pipeline.predict(X_test)
bigram_vocabulary_size = len(bigram_pipeline.named_steps["tfidf"].vocabulary_)
bigram_metrics = binary_metrics(y_test, bigram_test_predictions)

display(pd.DataFrame([
    {
        "Features": "Unigram",
        "Vocabulary size": baseline_vocabulary_size,
        "F1": baseline_metrics["F1"],
        "Training time seconds": baseline_training_time,
    },
    {
        "Features": "Unigram + bigram",
        "Vocabulary size": bigram_vocabulary_size,
        "F1": bigram_metrics["F1"],
        "Training time seconds": bigram_training_time,
    },
]).round(4))

## 10. Representation comparison and results.csv


In [ ]:
model_records = [
    {
        "Model": "TF-IDF + Logistic Regression",
        "Representation": "TF-IDF unigram",
        "Classifier": "Logistic Regression",
        "Training_time_seconds": baseline_training_time,
        "Vocabulary_size": baseline_vocabulary_size,
        "Feature_Dimension": baseline_vocabulary_size,
        **baseline_metrics,
    },
    {
        "Model": "Average Word2Vec + Logistic Regression",
        "Representation": "Average Word2Vec",
        "Classifier": "Logistic Regression",
        "Training_time_seconds": w2v_training_time,
        "Vocabulary_size": w2v_vocabulary_size,
        "Feature_Dimension": WORD2VEC_SIZE,
        **w2v_metrics,
    },
    {
        "Model": "TF-IDF + Linear SVM",
        "Representation": "TF-IDF unigram",
        "Classifier": "Linear SVM",
        "Training_time_seconds": svm_training_time,
        "Vocabulary_size": svm_vocabulary_size,
        "Feature_Dimension": svm_vocabulary_size,
        **svm_metrics,
    },
    {
        "Model": "TF-IDF unigram + bigram + Logistic Regression",
        "Representation": "TF-IDF unigram + bigram",
        "Classifier": "Logistic Regression",
        "Training_time_seconds": bigram_training_time,
        "Vocabulary_size": bigram_vocabulary_size,
        "Feature_Dimension": bigram_vocabulary_size,
        **bigram_metrics,
    },
]
results = pd.DataFrame(model_records)
results.to_csv(OUTPUT_DIR / "results.csv", index=False, float_format="%.6f")
display(results.sort_values("F1", ascending=False).round(4))

## 11. Confusion matrix



In [ ]:
cm = confusion_matrix(y_test, baseline_test_predictions, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
image = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.figure.colorbar(image, ax=ax)
ax.set(
    xticks=np.arange(2),
    yticks=np.arange(2),
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"],
    ylabel="Actual label",
    xlabel="Predicted label",
    title="Confusion Matrix — TF-IDF + Logistic Regression",
)
threshold_value = cm.max() / 2 if cm.size else 0
for row in range(cm.shape[0]):
    for column in range(cm.shape[1]):
        ax.text(column, row, format(cm[row, column], "d"), ha="center", va="center", color="white" if cm[row, column] > threshold_value else "black")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

display(pd.DataFrame([{
    "True positives": int(cm[1, 1]),
    "False positives": int(cm[0, 1]),
    "False negatives": int(cm[1, 0]),
    "True negatives": int(cm[0, 0]),
}]))

## 12. Word order and negation


In [ ]:
examples = [
    "this movie is good",
    "this movie is not good",
    "this movie is good not",
    "this movie is very good",
]
example_vectors = average_word2vec(examples, word2vec_model, WORD2VEC_SIZE)

def cosine_similarity(first, second):
    first_norm = np.linalg.norm(first)
    second_norm = np.linalg.norm(second)
    if first_norm == 0 or second_norm == 0:
        return 0.0
    return float(np.dot(first, second) / (first_norm * second_norm))

word_order_rows = [
    {
        "Sentence A": examples[0],
        "Sentence B": examples[1],
        "Cosine similarity": cosine_similarity(example_vectors[0], example_vectors[1]),
        "Interpretation": "Difference is the added negation token",
    },
    {
        "Sentence A": examples[1],
        "Sentence B": examples[2],
        "Cosine similarity": cosine_similarity(example_vectors[1], example_vectors[2]),
        "Interpretation": "Same tokens in a different order",
    },
    {
        "Sentence A": examples[0],
        "Sentence B": examples[3],
        "Cosine similarity": cosine_similarity(example_vectors[0], example_vectors[3]),
        "Interpretation": "Difference is the intensifier token",
    },
]
display(pd.DataFrame(word_order_rows).round(4))

## 13. Inspect Logistic Regression weights



In [ ]:
feature_names = baseline.named_steps["tfidf"].get_feature_names_out()
weights = baseline.named_steps["classifier"].coef_[0]
weight_order = np.argsort(weights)
negative_features = pd.DataFrame({
    "Feature": feature_names[weight_order[:15]],
    "Weight": weights[weight_order[:15]],
}).sort_values("Weight")
positive_features = pd.DataFrame({
    "Feature": feature_names[weight_order[-15:][::-1]],
    "Weight": weights[weight_order[-15:][::-1]],
}).sort_values("Weight", ascending=False)

display(positive_features.round(4).rename(columns={"Feature": "Top positive feature"}))
display(negative_features.round(4).rename(columns={"Feature": "Top negative feature"}))

## 14. Error analysis


In [ ]:
error_frame = pd.DataFrame({
    "Text": X_test.reset_index(drop=True),
    "True label": y_test.reset_index(drop=True),
    "Predicted label": pd.Series(baseline_test_predictions),
})
error_frame["Correct"] = error_frame["True label"] == error_frame["Predicted label"]

vectorizer = baseline.named_steps["tfidf"]
classifier = baseline.named_steps["classifier"]
feature_names = vectorizer.get_feature_names_out()
feature_weights = classifier.coef_[0]
training_vocabulary = set(vectorizer.vocabulary_)


def candidate_category(text):
    lowered = normalize_text(text).lower()
    tokens = simple_preprocess(lowered, deacc=True)
    if any(phrase in lowered for phrase in ["yeah right", "as if", "what a joke", "laughable", "so-called"]):
        return "sarcasm"
    if any(term in lowered for term in ["not ", "no ", "never", "neither", "nor ", "without", "isn't", "wasn't", "weren't", "didn't", "doesn't", "can't", "couldn't", "won't"]):
        return "negation"
    if len(tokens) <= 30:
        return "short text"
    content_tokens = [token for token in tokens if len(token) > 2]
    if content_tokens:
        out_of_vocabulary_ratio = sum(token not in training_vocabulary for token in content_tokens) / len(content_tokens)
        if out_of_vocabulary_ratio >= 0.25:
            return "rare word"
    if any(character.isdigit() for character in text):
        return "domain-specific term"
    return "lexical ambiguity"


def get_feature_evidence(text, direction, limit=3):
    row = vectorizer.transform([text])
    contributions = [
        (feature_names[index], float(value * feature_weights[index]))
        for index, value in zip(row.indices, row.data)
        if (value * feature_weights[index] > 0 if direction == "positive" else value * feature_weights[index] < 0)
    ]
    contributions.sort(key=lambda item: item[1], reverse=(direction == "positive"))
    return contributions[:limit]


def format_evidence(items):
    if not items:
        return "no strong matched feature"
    return ", ".join(f"{term} ({contribution:+.3f})" for term, contribution in items)


def explain_sample(row):
    true_label = label_names[int(row["True label"])]
    predicted_label = label_names[int(row["Predicted label"])]
    predicted_direction = "positive" if predicted_label == "positive" else "negative"
    true_direction = "positive" if true_label == "positive" else "negative"
    predicted_features = get_feature_evidence(row["Text"], predicted_direction)

    if row["Correct"]:
        return (
            f"Correct prediction. Features contributing toward {predicted_label}: "
            f"{format_evidence(predicted_features)}. Check the full review to confirm these terms "
            "express sentiment rather than incidental topic or plot information. Contributions "
            "describe the linear model score and do not prove causation."
        )

    true_features = get_feature_evidence(row["Text"], true_direction)
    return (
        f"Incorrect prediction: true label is {true_label}, but the model predicts {predicted_label}. "
        f"Features pushing toward the prediction: {format_evidence(predicted_features)}. "
        f"Features pushing toward the true label: {format_evidence(true_features)}. "
        "Read the full review to check whether negation, contrast, sarcasm, or wider context changes "
        "the sentiment. These feature contributions are evidence for inspection, not a confirmed cause."
    )


correct_count = int(error_frame["Correct"].sum())
incorrect_count = int((~error_frame["Correct"]).sum())
correct_examples = error_frame[error_frame["Correct"]].sample(n=min(3, correct_count), random_state=SEED)
incorrect_examples = error_frame[~error_frame["Correct"]].sample(n=min(3, incorrect_count), random_state=SEED)
selected_errors = pd.concat([correct_examples, incorrect_examples], axis=0).copy()
selected_errors["Analysis category"] = selected_errors["Text"].map(candidate_category)
selected_errors["Analysis"] = selected_errors.apply(explain_sample, axis=1)
selected_errors["True label"] = selected_errors["True label"].map(label_names)
selected_errors["Predicted label"] = selected_errors["Predicted label"].map(label_names)
selected_errors["Text excerpt"] = selected_errors["Text"].map(lambda text: text[:320] + ("…" if len(text) > 320 else ""))

output_columns = ["Text excerpt", "True label", "Predicted label", "Analysis category", "Analysis"]
with pd.option_context("display.max_colwidth", 240, "display.width", 220):
    display(selected_errors[output_columns])

markdown_lines = [
    "# Error Analysis",
    "",
    "Baseline: TF-IDF + Logistic Regression. Examples are drawn from the held-out test split.",
    "",
    "Three correct predictions and three incorrect predictions are sampled where enough examples are available.",
    "The category is an automatic candidate, not a confirmed diagnosis. Feature contributions show which TF-IDF terms affect the linear score; manually inspect the complete review before assigning a final explanation.",
    "",
    "| Text excerpt | True label | Predicted label | Candidate category | Analysis |",
    "|---|---|---|---|---|",
]
for _, row in selected_errors.iterrows():
    safe_values = [
        str(row["Text excerpt"]).replace("|", "\\|").replace("\n", " "),
        str(row["True label"]),
        str(row["Predicted label"]),
        str(row["Analysis category"]),
        str(row["Analysis"]).replace("|", "\\|").replace("\n", " "),
    ]
    markdown_lines.append("| " + " | ".join(safe_values) + " |")
Path("error_analysis.md").write_text("\n".join(markdown_lines) + "\n", encoding="utf-8")
